# Практическая работа № 3. Распределенная модель орбаботки данных

Выполнил: Савинский Владимир Олегович, ИКБО-40-23, 14.10.2026

**Источник:** NBA Basketball Database (Wyatt Walsh), таблицы `game`, `game_summary`, `team`;
репозиторий набора — https://github.com/wyattowalsh/nbadb, исходные данные — публичная статистика `stats.nba.com`.
**Лицензия:** CC BY-SA 4.0 (при распространении нужно сохранять атрибуцию).
**Персональные данные:** отсутствуют — только результаты матчей и справочник команд.

**Цель:** средствами Spark подготовить витрину матчей NBA и проверить, как менялись результативность игр,
трёхочковые броски и преимущество домашней площадки по десятилетиям.

**Связанные DataFrame:**
- `game` — основная таблица фактов, одна строка на матч (ключ `game_id`), 55 столбцов;
- `team` — справочник из 30 действующих команд (ключ `id` = `game.team_id_home`);
- `game_summary` — сводка по матчу: статус, код матча, сезон (ключ `game_id`).

In [1]:
# Шаг 0. Окружение (Windows) — то же, что в notebook по Steam
import os, sys
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ.pop("SPARK_HOME", None)   # используем PySpark из .venv, а не отдельную установку Spark
# если JAVA_HOME не задан в системе, раскомментируй:
# os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-21.0.12"
print("Python:", sys.version)
print("Интерпретатор:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

Python: 3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]
Интерпретатор: D:\spark_practice\.venv\Scripts\python.exe
JAVA_HOME: C:\Program Files\Java\jdk-21.0.12
HADOOP_HOME: C:\hadoop


## Шаг 1. Источник данных и способ чтения

Набор выгружен с Kaggle в виде CSV-файлов (папка `data3`). Spark читает CSV встроенными средствами,
поэтому промежуточная СУБД не нужна: таблицы читаются напрямую из CSV.
Это проще и воспроизводимее, чем загрузка в SQLite и чтение через JDBC:
не требуется внешний JAR-драйвер и его настройка в classpath (в методичке по той же причине выбран staging без JDBC).

Перед чтением проверяем, что все три файла на месте, и смотрим их заголовки.

In [2]:
from pathlib import Path
import time
import pandas as pd

DATA_DIR = Path("data3")
OUT = Path("output_nba")
CSV_FILES = {
    "game": DATA_DIR / "game.csv",
    "game_summary": DATA_DIR / "game_summary.csv",
    "team": DATA_DIR / "team.csv",
}

for name, path in CSV_FILES.items():
    print(f"{name:13} {'OK' if path.exists() else 'НЕ НАЙДЕН'}  {path.resolve()}")
missing = [str(p) for p in CSV_FILES.values() if not p.exists()]
assert not missing, f"Не найдены файлы: {missing}"

for name, path in CSV_FILES.items():
    header = pd.read_csv(path, nrows=0).columns.tolist()
    print(f"\n{name}: {len(header)} столбцов, {path.stat().st_size / 1e6:.1f} МБ")
    print(header)

game          OK  D:\spark_practice\data3\game.csv
game_summary  OK  D:\spark_practice\data3\game_summary.csv
team          OK  D:\spark_practice\data3\team.csv

game: 55 столбцов, 20.3 МБ
['season_id', 'team_id_home', 'team_abbreviation_home', 'team_name_home', 'game_id', 'game_date', 'matchup_home', 'wl_home', 'min', 'fgm_home', 'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home', 'fg3_pct_home', 'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home', 'dreb_home', 'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home', 'pf_home', 'pts_home', 'plus_minus_home', 'video_available_home', 'team_id_away', 'team_abbreviation_away', 'team_name_away', 'matchup_away', 'wl_away', 'fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away', 'fg3_pct_away', 'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away', 'dreb_away', 'reb_away', 'ast_away', 'stl_away', 'blk_away', 'tov_away', 'pf_away', 'pts_away', 'plus_minus_away', 'video_available_away', 'season_type']

game_summary: 14 столбцов, 5.6 М

**Вывод:** все три CSV на месте. `game` — широкая таблица фактов (55 столбцов, 20,3 МБ: очки, броски, подборы и т. д. для хозяев и гостей),
`game_summary` — сводка по матчам (14 столбцов, 5,6 МБ), `team` — маленький справочник команд (7 столбцов).

## Шаг 2. SparkSession и версии

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("Practice3_NBA")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Python:", sys.version.split()[0])
print("Java:", spark.sparkContext._jvm.java.lang.System.getProperty("java.version"))
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Ядер (defaultParallelism):", spark.sparkContext.defaultParallelism)
print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("spark.sql.ansi.enabled:", spark.conf.get("spark.sql.ansi.enabled"))

D:\spark_practice\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Python: 3.11.0
Java: 21.0.12
Spark: 4.2.0
Master: local[*]
Ядер (defaultParallelism): 16
spark.sql.shuffle.partitions: 8
spark.sql.ansi.enabled: true


## Шаг 3. Чтение основной таблицы без схемы

Без `inferSchema` все поля CSV читаются как `string`.

In [4]:
game_raw = (
    spark.read
    .option("header", True)
    .option("escape", '"')
    .csv(str(CSV_FILES["game"]))
)
print("Строк:", game_raw.count())
print("Столбцов:", len(game_raw.columns))
print(game_raw.columns)
game_raw.printSchema()
print("Разделов (partitions):", game_raw.rdd.getNumPartitions())
game_raw.select("game_id", "season_id", "game_date", "team_name_home", "team_name_away",
                "pts_home", "pts_away", "wl_home", "season_type").show(5, truncate=False)

Строк: 65698
Столбцов: 55
['season_id', 'team_id_home', 'team_abbreviation_home', 'team_name_home', 'game_id', 'game_date', 'matchup_home', 'wl_home', 'min', 'fgm_home', 'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home', 'fg3_pct_home', 'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home', 'dreb_home', 'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home', 'pf_home', 'pts_home', 'plus_minus_home', 'video_available_home', 'team_id_away', 'team_abbreviation_away', 'team_name_away', 'matchup_away', 'wl_away', 'fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away', 'fg3_pct_away', 'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away', 'dreb_away', 'reb_away', 'ast_away', 'stl_away', 'blk_away', 'tov_away', 'pf_away', 'pts_away', 'plus_minus_away', 'video_available_away', 'season_type']
root
 |-- season_id: string (nullable = true)
 |-- team_id_home: string (nullable = true)
 |-- team_abbreviation_home: string (nullable = true)
 |-- team_name_home: string (nullable = true)
 |-- g

**Вывод:** Spark прочитал 65 698 строк и 55 столбцов; без схемы все поля имеют тип `string`, поэтому сложить очки или сравнить даты пока нельзя.
Файл разбит на 5 разделов (partitions), их обрабатывают параллельно.
В исходнике `game_id` записан с ведущими нулями (`0024600001`); при приведении к числу нули отбрасываются, это не мешает соединению таблиц.

## Шаг 4. Явная схема StructType и сравнение с inferSchema

Схема строится явно по смыслу столбцов:
- идентификаторы (`game_id`, `season_id`, `team_id_*`) — `LongType`;
- названия, аббревиатуры, результат `W/L`, тип сезона и дата — `StringType` (дата разбирается на шаге 5);
- вся игровая статистика — `DoubleType`: в старых матчах часть значений пустая, а в CSV числа могут быть записаны как `25.0`.
  С `IntegerType` такие значения Spark молча превратил бы в `null`.

In [5]:
LONG_COLS = {"season_id", "team_id_home", "team_id_away", "game_id"}
STRING_COLS = {
    "team_abbreviation_home", "team_name_home", "game_date", "matchup_home", "wl_home",
    "team_abbreviation_away", "team_name_away", "matchup_away", "wl_away", "season_type",
}

def field_type(col):
    if col in LONG_COLS:
        return T.LongType()
    if col in STRING_COLS:
        return T.StringType()
    return T.DoubleType()

game_schema = T.StructType([T.StructField(c, field_type(c), True) for c in game_raw.columns])

t0 = time.time()
game_typed = (spark.read.option("header", True).option("escape", '"')
              .schema(game_schema).csv(str(CSV_FILES["game"])))
n_explicit = game_typed.count()
t_explicit = time.time() - t0

t0 = time.time()
game_inferred = (spark.read.option("header", True).option("escape", '"')
                 .option("inferSchema", True).csv(str(CSV_FILES["game"])))
n_inferred = game_inferred.count()
t_inferred = time.time() - t0

print(f"Явная схема:  {n_explicit} строк, {t_explicit:.2f} с")
print(f"inferSchema:  {n_inferred} строк, {t_inferred:.2f} с (лишний проход по файлу для вывода типов)")
game_typed.printSchema()

cmp = pd.DataFrame({
    "column": game_typed.columns,
    "explicit": [f.dataType.simpleString() for f in game_typed.schema],
    "inferSchema": [game_inferred.schema[c].dataType.simpleString() for c in game_typed.columns],
})
print("Столбцы, где типы разошлись:")
cmp[cmp["explicit"] != cmp["inferSchema"]]

Явная схема:  65698 строк, 1.97 с
inferSchema:  65698 строк, 5.73 с (лишний проход по файлу для вывода типов)
root
 |-- season_id: long (nullable = true)
 |-- team_id_home: long (nullable = true)
 |-- team_abbreviation_home: string (nullable = true)
 |-- team_name_home: string (nullable = true)
 |-- game_id: long (nullable = true)
 |-- game_date: string (nullable = true)
 |-- matchup_home: string (nullable = true)
 |-- wl_home: string (nullable = true)
 |-- min: double (nullable = true)
 |-- fgm_home: double (nullable = true)
 |-- fga_home: double (nullable = true)
 |-- fg_pct_home: double (nullable = true)
 |-- fg3m_home: double (nullable = true)
 |-- fg3a_home: double (nullable = true)
 |-- fg3_pct_home: double (nullable = true)
 |-- ftm_home: double (nullable = true)
 |-- fta_home: double (nullable = true)
 |-- ft_pct_home: double (nullable = true)
 |-- oreb_home: double (nullable = true)
 |-- dreb_home: double (nullable = true)
 |-- reb_home: double (nullable = true)
 |-- ast_home:

,column,explicit,inferSchema
0,season_id,bigint,int
1,team_id_home,bigint,int
4,game_id,bigint,int
5,game_date,string,timestamp
8,min,double,int
27,plus_minus_home,double,int
28,video_available_home,double,int
29,team_id_away,bigint,int
52,plus_minus_away,double,int
53,video_available_away,double,int


In [6]:
# Проверка: явная схема не потеряла значения (непустая строка не превратилась в null)
check = ["game_id", "season_id", "pts_home", "pts_away", "fg3_pct_home", "plus_minus_home"]
lost = {c: game_raw.filter(F.col(c).isNotNull() & (F.trim(F.col(c)) != "")).count()
           - game_typed.filter(F.col(c).isNotNull()).count()
        for c in check}
print("Значений потеряно при приведении типов:", lost)
assert all(v == 0 for v in lost.values()), "Часть значений не подошла под тип схемы"

Значений потеряно при приведении типов: {'game_id': 0, 'season_id': 0, 'pts_home': 0, 'pts_away': 0, 'fg3_pct_home': 0, 'plus_minus_home': 0}


**Вывод:** чтение с явной схемой заняло 1,97 с, с `inferSchema` — 5,73 с, то есть почти в 3 раза дольше из-за лишнего прохода по файлу.
Типы разошлись в 10 столбцах:
- идентификаторы `inferSchema` сделал `int`, в явной схеме — `long`: id команд (около 1,6 млрд) близки к пределу `int` (2,1 млрд), `long` надёжнее;
- `game_date` `inferSchema` сам распознал как `timestamp`, в явной схеме дата оставлена строкой и разбирается на шаге 5;
- `min`, `plus_minus_*`, `video_available_*` `inferSchema` сделал `int`, в явной схеме — `double`, как и вся игровая статистика.

Проверка показала, что при приведении типов ни одно значение не потерялось: схема подходит к данным.

## Шаг 5. Transformations и вычисляемые признаки

Используются `dropDuplicates`, `withColumn`, `filter`, `select`, `orderBy`.

Вычисляемые признаки:
- `game_ts` — дата матча как timestamp (в файле строка `1946-11-01 00:00:00`);
- `season_year`, `decade` — год начала сезона и десятилетие (`season_id = 21946`: первая цифра — тип сезона, последние четыре — год);
- `total_points`, `point_difference` — суммарные очки и разница очков;
- `home_win` — 1, если хозяева выиграли, 0 — проиграли, `null` — результат неизвестен.

Фильтр по типу сезона: оставляем регулярный чемпионат и плей-офф.
Матчи всех звёзд и предсезонные — показательные игры: состав команд и мотивация в них другие, они исказили бы средние показатели.

In [7]:
n_raw = game_typed.count()
game_dedup = game_typed.dropDuplicates(["game_id"])
n_dedup = game_dedup.count()
print(f"Строк: {n_raw}, после удаления повторов game_id: {n_dedup} (удалено {n_raw - n_dedup})")

print("Типы сезона до фильтра:")
game_dedup.groupBy("season_type").count().orderBy(F.desc("count")).show(truncate=False)

games = (
    game_dedup
    .withColumn("game_ts", F.coalesce(
        F.try_to_timestamp(F.col("game_date"), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.col("game_date"), F.lit("yyyy-MM-dd"))))
    .withColumn("season_year", (F.col("season_id") % 10000).cast("int"))
    .withColumn("decade", (F.floor(F.col("season_year") / 10) * 10).cast("int"))
    .withColumn("total_points", F.col("pts_home") + F.col("pts_away"))
    .withColumn("point_difference", F.col("pts_home") - F.col("pts_away"))
    .withColumn("home_win", F.when(F.col("wl_home") == "W", 1).when(F.col("wl_home") == "L", 0))
    .filter(F.col("season_type").isin("Regular Season", "Playoffs"))
    .select(
        "game_id", "season_id", "season_year", "decade", "season_type", "game_ts",
        "team_id_home", "team_abbreviation_home", "team_name_home",
        "team_id_away", "team_name_away", "wl_home", "home_win",
        "pts_home", "pts_away", "total_points", "point_difference",
        "fg_pct_home", "fg3_pct_home", "fg3a_home", "ft_pct_home",
        "reb_home", "ast_home", "tov_home",
    )
)
print("Строк после фильтра по типу сезона:", games.count())
games.printSchema()

(games
 .select("game_id", "game_ts", "season_type", "team_name_home", "team_name_away",
         "pts_home", "pts_away", "total_points", "home_win")
 .orderBy(F.desc("total_points"))
 .show(10, truncate=False))

Строк: 65698, после удаления повторов game_id: 65642 (удалено 56)
Типы сезона до фильтра:
+--------------+-----+
|season_type   |count|
+--------------+-----+
|Regular Season|60192|
|Playoffs      |3842 |
|Pre Season    |1536 |
|All-Star      |63   |
|All Star      |9    |
+--------------+-----+

Строк после фильтра по типу сезона: 64034
root
 |-- game_id: long (nullable = true)
 |-- season_id: long (nullable = true)
 |-- season_year: integer (nullable = true)
 |-- decade: integer (nullable = true)
 |-- season_type: string (nullable = true)
 |-- game_ts: timestamp (nullable = true)
 |-- team_id_home: long (nullable = true)
 |-- team_abbreviation_home: string (nullable = true)
 |-- team_name_home: string (nullable = true)
 |-- team_id_away: long (nullable = true)
 |-- team_name_away: string (nullable = true)
 |-- wl_home: string (nullable = true)
 |-- home_win: integer (nullable = true)
 |-- pts_home: double (nullable = true)
 |-- pts_away: double (nullable = true)
 |-- total_points: do

**Вывод:** удалено 56 повторов `game_id` (осталось 65 642 матча). Фильтр по типу сезона отбросил 1 608 показательных матчей:
1 536 предсезонных и 72 матча всех звёзд. Тип «матч всех звёзд» в данных записан двумя способами — `All-Star` (63) и `All Star` (9);
фильтр по списку нужных типов отбрасывает оба варианта. В анализе осталось 64 034 матча: 60 192 регулярного сезона и 3 842 плей-офф.

Самый результативный матч — Denver Nuggets против Detroit Pistons 13.12.1983, 184:186, всего 370 очков. Это рекорд NBA, матч шёл с тремя овертаймами.
На втором месте — LA Clippers против Sacramento Kings 24.02.2023, 175:176 (351 очко).
Сами transformations выполняются мгновенно: Spark только строит план, а вычисления запускаются на `count()` и `show()` (lazy evaluation).

## Шаг 6. Пропуски и их предметный смысл

Считаем пропуски в семи полях, затем смотрим, как доля пропусков распределена по десятилетиям.

In [8]:
null_cols = ["pts_home", "pts_away", "wl_home", "fg3_pct_home", "ast_home", "reb_home", "game_ts"]
total = games.count()
nulls = games.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in null_cols]).first().asDict()
nulls = {k: (v or 0) for k, v in nulls.items()}
pd.DataFrame({
    "missing": nulls,
    "pct": {k: round(v / total * 100, 2) if total else None for k, v in nulls.items()},
})

,missing,pct
pts_home,0,0.00
pts_away,0,0.00
wl_home,0,0.00
fg3_pct_home,19021,29.70
ast_home,15805,24.68
reb_home,15729,24.56
game_ts,0,0.00


In [9]:
# Доля пропусков по десятилетиям: пропуски — историческая особенность, а не ошибка
(games.groupBy("decade")
 .agg(F.count("*").alias("games"),
      F.round(F.avg(F.col("fg3_pct_home").isNull().cast("int")) * 100, 1).alias("fg3_missing_pct"),
      F.round(F.avg(F.col("ast_home").isNull().cast("int")) * 100, 1).alias("ast_missing_pct"),
      F.round(F.avg(F.col("reb_home").isNull().cast("int")) * 100, 1).alias("reb_missing_pct"))
 .orderBy("decade")
 .show())

+------+-----+---------------+---------------+---------------+
|decade|games|fg3_missing_pct|ast_missing_pct|reb_missing_pct|
+------+-----+---------------+---------------+---------------+
|  1940| 1538|          100.0|           99.6|          100.0|
|  1950| 3283|          100.0|           98.9|           97.3|
|  1960| 3303|          100.0|           99.2|           98.6|
|  1970| 5996|           99.9|           99.0|           99.0|
|  1980|10319|           47.2|           17.6|           17.5|
|  1990|11515|            0.3|            0.0|            0.0|
|  2000|12793|            0.0|            0.0|            0.0|
|  2010|11491|            0.0|            0.0|            0.0|
|  2020| 3796|            0.0|            0.0|            0.0|
+------+-----+---------------+---------------+---------------+



**Решения по пропускам (строки только из-за null не удаляются):**

| Поле | Действие | Причина |
|---|---|---|
| `fg3_pct_home` | оставить `null`, не заполнять | трёхочковая линия появилась в NBA только в сезоне 1979/80; до этого бросков за три очка не было, 0 исказил бы средние |
| `ast_home`, `reb_home` | оставить `null` | в ранние годы эту статистику не вели; `avg()` в Spark пропускает `null`, поэтому средние считаются только по известным значениям |
| `pts_home`, `pts_away` | для `total_points` оставить только матчи с обоими значениями | без счёта нельзя посчитать результативность; в данных таких матчей не оказалось, фильтр оставлен как защита |
| `wl_home` | `home_win = null` | исход неизвестен; заполнение 0 занизило бы долю побед хозяев |

In [10]:
games_clean = games.filter(F.col("pts_home").isNotNull() & F.col("pts_away").isNotNull())
print("Строк до:", games.count(), " после фильтра по счёту:", games_clean.count())

Строк до: 64034  после фильтра по счёту: 64034


**Вывод:** счёт и результат матча заполнены у всех 64 034 матчей, фильтр по счёту не удалил ни одной строки.
Самые неполные поля — процент трёхочковых (пропуск у 29,7% матчей), передачи (24,7%) и подборы (24,6%).
Таблица по десятилетиям показывает, что это не ошибка данных, а отсутствие учёта:
до 1980-х эти показатели пусты почти в 100% матчей, в 1980-х — частично (трёхочковые — 47%, передачи и подборы — около 18%),
а с 1990-х пропусков практически нет.

## Шаг 7. Агрегации

1. По типу сезона — 6 агрегатных функций.
2. По десятилетиям — динамика результативности, трёхочковых и преимущества домашней площадки.

In [11]:
season_agg = (
    games_clean.groupBy("season_type")
    .agg(
        F.count("*").alias("games"),
        F.round(F.avg("pts_home"), 2).alias("avg_home_points"),
        F.round(F.avg("pts_away"), 2).alias("avg_away_points"),
        F.round(F.avg("total_points"), 2).alias("avg_total_points"),
        F.max("total_points").alias("max_total_points"),
        F.round(F.avg("home_win") * 100, 2).alias("home_win_pct"),
    )
    .orderBy(F.desc("games"))
)
season_agg.show(truncate=False)

+--------------+-----+---------------+---------------+----------------+----------------+------------+
|season_type   |games|avg_home_points|avg_away_points|avg_total_points|max_total_points|home_win_pct|
+--------------+-----+---------------+---------------+----------------+----------------+------------+
|Regular Season|60192|104.76         |101.16         |205.92          |370.0           |61.84       |
|Playoffs      |3842 |103.22         |98.79          |202.01          |304.0           |64.45       |
+--------------+-----+---------------+---------------+----------------+----------------+------------+



In [12]:
decade_agg = (
    games_clean.groupBy("decade")
    .agg(
        F.count("*").alias("games"),
        F.round(F.avg("total_points"), 1).alias("avg_total_points"),
        F.round(F.avg("fg3a_home"), 1).alias("avg_fg3_attempts_home"),
        F.round(F.avg("fg3_pct_home"), 3).alias("avg_fg3_pct_home"),
        F.round(F.avg("home_win") * 100, 1).alias("home_win_pct"),
    )
    .orderBy("decade")
)
decade_agg.show()

+------+-----+----------------+---------------------+----------------+------------+
|decade|games|avg_total_points|avg_fg3_attempts_home|avg_fg3_pct_home|home_win_pct|
+------+-----+----------------+---------------------+----------------+------------+
|  1940| 1538|           152.5|                  0.0|            NULL|        65.0|
|  1950| 3283|           188.7|                  0.0|            NULL|        73.1|
|  1960| 3303|           228.1|                 NULL|            NULL|        64.5|
|  1970| 5996|           215.2|                  1.8|           0.083|        64.0|
|  1980|10319|           217.8|                  5.0|           0.296|        64.8|
|  1990|11515|           200.3|                 12.2|           0.344|        61.3|
|  2000|12793|           194.2|                 16.2|           0.355|        60.6|
|  2010|11491|           207.3|                 25.2|           0.357|        58.5|
|  2020| 3796|           224.5|                 34.8|           0.362|      

**Вывод:**
- **Плей-офф и регулярный сезон.** В плей-офф хозяева выигрывают чаще: 64,5% матчей против 61,8% в регулярном сезоне.
  Плей-офф при этом менее результативен: в среднем 202 очка за матч против 206.
- **Результативность по десятилетиям.** Она менялась волнами: 152,5 очка в 1940-х, пик 228,1 в 1960-х,
  спад до 194,2 в 2000-х и снова рост до 224,5 в 2020-х.
- **Трёхочковые.** Хозяева делают в среднем 5 трёхочковых попыток за матч в 1980-х и 34,8 в 2020-х, то есть в 7 раз больше.
  Точность выросла с 29,6% до 36,2%. Значения для 1940–1970-х не показательны: трёхочковой линии ещё не было,
  а попытки за 1970-е есть лишь в единичных матчах (99,9% пропусков).
- **Преимущество домашней площадки.** Оно сохраняется во все десятилетия (больше 50%), но постепенно слабеет:
  73,1% побед хозяев в 1950-х, 55,8% в 2020-х.

## Шаг 8. JOIN: LEFT и INNER

**LEFT JOIN** матчей со справочником `team` по `team_id_home = id`. Справочник содержит только 30 действующих команд,
поэтому исторические клубы (например, Baltimore Bullets) соответствия не найдут — LEFT JOIN сохраняет такие матчи.

**INNER JOIN** со сводкой `game_summary` по `game_id` — оставляет только матчи, для которых есть сводка.

In [13]:
team_df = spark.read.option("header", True).option("inferSchema", True).csv(str(CSV_FILES["team"]))
if "d" in team_df.columns and "id" not in team_df.columns:   # в некоторых копиях заголовок id испорчен
    team_df = team_df.withColumnRenamed("d", "id")
team_lookup = team_df.select(
    F.col("id").cast("long").alias("team_id"),
    F.col("full_name").alias("team_full_name"),
    F.col("city").alias("team_city"),
    F.col("year_founded").cast("int").alias("team_year_founded"),
)
print("Команд в справочнике:", team_lookup.count(),
      " уникальных id:", team_lookup.select("team_id").distinct().count())

n_before = games_clean.count()
games_team = games_clean.join(
    team_lookup, games_clean["team_id_home"] == team_lookup["team_id"], "left"
).drop("team_id")
n_after = games_team.count()
n_unmatched = games_team.filter(F.col("team_full_name").isNull()).count()
print(f"LEFT JOIN: до {n_before}, после {n_after}, без соответствия в справочнике {n_unmatched}")

print("Домашние команды без соответствия:")
(games_team.filter(F.col("team_full_name").isNull())
 .groupBy("team_name_home").count().orderBy(F.desc("count")).show(10, truncate=False))

Команд в справочнике: 30  уникальных id: 30
LEFT JOIN: до 64034, после 64034, без соответствия в справочнике 1129
Домашние команды без соответствия:
+-----------------------+-----+
|team_name_home         |count|
+-----------------------+-----+
|Baltimore Bullets      |224  |
|Washington Capitols    |149  |
|Indianapolis Olympians |148  |
|Chicago Stags          |126  |
|St. Louis Bombers      |125  |
|Providence Steamrollers|79   |
|Anderson Packers       |38   |
|Waterloo Hawks         |32   |
|Sheboygan Redskins     |32   |
|Cleveland Rebels       |31   |
+-----------------------+-----+
only showing top 10 rows


In [14]:
summary_raw = (spark.read.option("header", True).option("escape", '"')
               .option("inferSchema", True).csv(str(CSV_FILES["game_summary"])))
summary = summary_raw.select(
    F.col("game_id").cast("long").alias("game_id"),
    F.col("gamecode"),
    F.col("game_status_id").cast("int").alias("game_status_id"),
    F.col("live_period").cast("int").alias("periods"),
)
n_sum, n_sum_ids = summary.count(), summary.select("game_id").distinct().count()
print(f"game_summary: {n_sum} строк, уникальных game_id {n_sum_ids} — повторы размножили бы строки при JOIN")
summary_one = summary.dropDuplicates(["game_id"])

games_summary = games_team.join(summary_one, "game_id", "inner")
print(f"INNER JOIN: до {games_team.count()}, после {games_summary.count()}, "
      f"уникальных game_id {games_summary.select('game_id').distinct().count()}")

# Сколько периодов длились матчи: 4 — основное время, больше 4 — овертаймы
(games_summary.groupBy("periods").count().orderBy("periods").show())

game_summary: 58110 строк, уникальных game_id 58021 — повторы размножили бы строки при JOIN
INNER JOIN: до 64034, после 56554, уникальных game_id 56554
+-------+-----+
|periods|count|
+-------+-----+
|      0|  154|
|      4|53186|
|      5| 2742|
|      6|  393|
|      7|   63|
|      8|   15|
|      9|    1|
+-------+-----+



**Вывод:**
- **LEFT JOIN** со справочником сохранил все 64 034 матча. У 1 129 из них (1,8%) хозяин не найден в справочнике.
  Это исторические клубы: Baltimore Bullets (224 матча), Washington Capitols (149), Indianapolis Olympians (148) и другие.
  INNER JOIN потерял бы эти матчи.
- **Повторы в сводке.** В `game_summary` 58 110 строк, но только 58 021 уникальный `game_id`. Повторы убраны до соединения,
  поэтому в результате каждый матч встречается один раз.
- **INNER JOIN** со сводкой оставил 56 554 матча: у 7 480 матчей (11,7%) сводки нет.
- **Овертаймы.** По сводке 53 186 матчей закончились в основное время (4 периода), 3 214 (5,7%) — с овертаймами.
  Самый длинный матч длился 9 периодов, то есть 5 овертаймов. У 154 матчей указано 0 периодов: это ошибка заполнения сводки.

## Шаг 9. Spark SQL

Временное представление `nba_games` и два запроса; первый сверяется с DataFrame API.

In [15]:
games_team.createOrReplaceTempView("nba_games")

# Запрос 1: действующие команды по доле домашних побед (GROUP BY + сортировка по агрегату)
sql_teams = spark.sql('''
SELECT team_full_name,
       COUNT(*)                          AS home_games,
       ROUND(AVG(home_win) * 100, 1)     AS home_win_pct,
       ROUND(AVG(point_difference), 2)   AS avg_point_diff
FROM nba_games
WHERE team_full_name IS NOT NULL
GROUP BY team_full_name
ORDER BY home_win_pct DESC
''')
sql_teams.show(10, truncate=False)

api_teams = (
    games_team.filter(F.col("team_full_name").isNotNull())
    .groupBy("team_full_name")
    .agg(F.count("*").alias("home_games"),
         F.round(F.avg("home_win") * 100, 1).alias("home_win_pct"),
         F.round(F.avg("point_difference"), 2).alias("avg_point_diff"))
)
same = sql_teams.exceptAll(api_teams).count() == 0 and api_teams.exceptAll(sql_teams).count() == 0
print("Результаты SQL и DataFrame API совпадают:", same)

+----------------------+----------+------------+--------------+
|team_full_name        |home_games|home_win_pct|avg_point_diff|
+----------------------+----------+------------+--------------+
|Los Angeles Lakers    |3027      |71.7        |6.47          |
|San Antonio Spurs     |1989      |71.5        |6.8           |
|Boston Celtics        |3074      |70.5        |6.04          |
|Utah Jazz             |1974      |68.5        |6.16          |
|Portland Trail Blazers|2097      |66.6        |5.17          |
|Phoenix Suns          |2202      |65.9        |5.16          |
|Oklahoma City Thunder |2247      |64.7        |4.72          |
|Atlanta Hawks         |2791      |64.6        |3.74          |
|Milwaukee Bucks       |2207      |63.7        |4.74          |
|Philadelphia 76ers    |2884      |63.7        |3.88          |
+----------------------+----------+------------+--------------+
only showing top 10 rows
Результаты SQL и DataFrame API совпадают: True


In [16]:
# Запрос 2: самые результативные матчи плей-офф
spark.sql('''
SELECT season_year, game_ts, team_name_home, team_name_away,
       pts_home, pts_away, total_points
FROM nba_games
WHERE season_type = 'Playoffs'
ORDER BY total_points DESC
LIMIT 10
''').show(truncate=False)

+-----------+-------------------+----------------------+----------------------+--------+--------+------------+
|season_year|game_ts            |team_name_home        |team_name_away        |pts_home|pts_away|total_points|
+-----------+-------------------+----------------------+----------------------+--------+--------+------------+
|1991       |1992-05-11 00:00:00|Phoenix Suns          |Portland Trail Blazers|151.0   |153.0   |304.0       |
|2020       |2021-06-01 00:00:00|Denver Nuggets        |Portland Trail Blazers|147.0   |140.0   |287.0       |
|1982       |1983-04-26 00:00:00|San Antonio Spurs     |Denver Nuggets        |152.0   |133.0   |285.0       |
|1989       |1990-04-28 00:00:00|Boston Celtics        |New York Knicks       |157.0   |128.0   |285.0       |
|1986       |1987-04-23 00:00:00|Dallas Mavericks      |Seattle SuperSonics   |151.0   |129.0   |280.0       |
|1966       |1967-04-01 00:00:00|San Francisco Warriors|St. Louis Hawks       |143.0   |136.0   |279.0       |
|

**Вывод:** лучше всех дома играют Los Angeles Lakers (71,7% побед в 3 027 домашних матчах), San Antonio Spurs (71,5%)
и Boston Celtics (70,5%); у них же самая большая средняя разница очков (+6–7 за матч).
Самый результативный матч плей-офф — Phoenix Suns против Portland Trail Blazers 11.05.1992, 151:153 (304 очка).
SQL и DataFrame API дали одинаковый результат — оба плана строит один оптимизатор Spark SQL.

## Шаг 10–11. Запись в Parquet и CSV, повторное чтение

- итоговая витрина `games_team` (матчи + справочник команд) — Parquet с разбиением по десятилетиям;
- небольшой агрегат `decade_agg` — CSV одним файлом через `coalesce(1)`.

In [17]:
OUT.mkdir(exist_ok=True)
games_team.write.mode("overwrite").partitionBy("decade").parquet(str(OUT / "nba_games_parquet"))
(decade_agg.coalesce(1).write.mode("overwrite")
    .option("header", True).csv(str(OUT / "decade_agg_csv")))

print("Подкаталоги Parquet:", sorted(p.name for p in (OUT / "nba_games_parquet").iterdir() if p.is_dir()))
print("Файлы CSV:", sorted(p.name for p in (OUT / "decade_agg_csv").iterdir()))

Подкаталоги Parquet: ['decade=1940', 'decade=1950', 'decade=1960', 'decade=1970', 'decade=1980', 'decade=1990', 'decade=2000', 'decade=2010', 'decade=2020']
Файлы CSV: ['._SUCCESS.crc', '.part-00000-b31dcdaa-21b0-4a9e-b362-943ded108051-c000.csv.crc', '_SUCCESS', 'part-00000-b31dcdaa-21b0-4a9e-b362-943ded108051-c000.csv']


In [18]:
games_back = spark.read.parquet(str(OUT / "nba_games_parquet"))
print("Строк до записи:", games_team.count(), " после чтения:", games_back.count())

def names_types(df):
    return sorted((f.name, f.dataType.simpleString()) for f in df.schema)

print("Имена и типы столбцов совпадают:", names_types(games_back) == names_types(games_team))
games_back.printSchema()   # decade — столбец разбиения, при чтении он оказывается в конце

Строк до записи: 64034  после чтения: 64034
Имена и типы столбцов совпадают: True
root
 |-- game_id: long (nullable = true)
 |-- season_id: long (nullable = true)
 |-- season_year: integer (nullable = true)
 |-- season_type: string (nullable = true)
 |-- game_ts: timestamp (nullable = true)
 |-- team_id_home: long (nullable = true)
 |-- team_abbreviation_home: string (nullable = true)
 |-- team_name_home: string (nullable = true)
 |-- team_id_away: long (nullable = true)
 |-- team_name_away: string (nullable = true)
 |-- wl_home: string (nullable = true)
 |-- home_win: integer (nullable = true)
 |-- pts_home: double (nullable = true)
 |-- pts_away: double (nullable = true)
 |-- total_points: double (nullable = true)
 |-- point_difference: double (nullable = true)
 |-- fg_pct_home: double (nullable = true)
 |-- fg3_pct_home: double (nullable = true)
 |-- fg3a_home: double (nullable = true)
 |-- ft_pct_home: double (nullable = true)
 |-- reb_home: double (nullable = true)
 |-- ast_home: 

**Вывод:** витрина записана в Parquet в 9 подкаталогов, по одному на десятилетие (`decade=1940` … `decade=2020`).
Агрегат по десятилетиям сохранён в CSV одним part-файлом. После повторного чтения число строк совпало (64 034),
имена и типы всех столбцов тоже. Столбец разбиения `decade` при чтении оказывается в конце схемы.

## Шаг 12. Итоговые выводы и ограничения

**Выводы:**
1. **Объём.** Из 65 698 строк таблицы `game` после удаления 56 повторов и 1 608 показательных матчей в анализе осталось
   64 034 матча NBA с 1946 по 2020-е годы.
2. **Пропуски.** Они исторические: процент трёхочковых, передачи и подборы до 1980-х почти не учитывались,
   поэтому пропуски не заполнялись нулями, а средние считались только по известным значениям.
3. **Результативность.** Она менялась волнами: 152 очка за матч в 1940-х, 228 в 1960-х, спад до 194 в 2000-х и рост до 225 в 2020-х.
4. **Трёхочковые.** Игра стала «трёхочковой»: число попыток хозяев выросло с 5 за матч в 1980-х до 34,8 в 2020-х,
   точность — с 29,6% до 36,2%.
5. **Домашняя площадка.** Преимущество есть всегда: хозяева выигрывают 61,8% матчей регулярного сезона и 64,5% в плей-офф.
   Но оно ослабевает: с 73,1% в 1950-х до 55,8% в 2020-х.
6. **Соединения.** Тип соединения меняет результат: LEFT JOIN сохранил 1 129 матчей исторических клубов (1,8%),
   а INNER JOIN со сводкой потерял 7 480 матчей (11,7%), для которых сводки нет.

**Ограничения:**
1. **Справочник команд.** `team` содержит только 30 действующих команд, поэтому анализ по командам возможен только для них;
   исторические франшизы остались без названия из справочника.
2. **Неполнота ранних лет.** Статистика до 1980-х неполная: сравнивать трёхочковые, передачи и подборы за всю историю нельзя.
3. **Качество данных.** Тип сезона «матч всех звёзд» записан двумя способами, у 154 матчей в сводке 0 периодов.
   Перед анализом такие значения нужно проверять.
4. **Режим запуска.** Работа выполнена в режиме `local[*]` на одном компьютере, это не проверка масштабируемости на кластере.

In [19]:
spark.stop()